In [ ]:
import requests
from rich import print

response = requests.get("https://www.tradingview.com/heatmap/stock/#%7B%22dataSource%22%3A%22SPX500%22%2C%22blockColor%22%3A%22change%22%2C%22blockSize%22%3A%22market_cap_basic%22%2C%22grouping%22%3A%22sector%22%7D")

print(response)

In [ ]:
import os

from dotenv import load_dotenv
from google.genai import Client, types

load_dotenv()


def get_web() -> str:
    """Get the ETF data for a given ETF symbol.

    It uses the Gemini URL context tool to get the data from the ETF Database website.

    Gemini does not support structured output when using the URL context tool, so it is separated into two steps.
    """
    client = Client(
        api_key=os.getenv("GEMINI_API_KEY"),
        http_options={"timeout": 600000},  # 10 minutes timeout
    )

    # Step 1: Get raw data using URL context tool
    url_context_tool = types.Tool(url_context=types.UrlContext())

    raw_response = client.models.generate_content(
        model=os.getenv("FAST_LLM"),
        contents=f"""Ehttps://www.tradingview.com/heatmap/stock/#%7B%22dataSource%22%3A%22SPX500%22%2C%22blockColor%22%3A%22change%22%2C%22blockSize%22%3A%22market_cap_basic%22%2C%22grouping%22%3A%22sector%22%7D What do you see? Can you list some names and numbers you see?""",
        config=types.GenerateContentConfig(
            tools=[url_context_tool],
            temperature=0,
            response_modalities=["TEXT"],
        ),
    )

    return raw_response


get_web()

In [ ]:
response.text

In [ ]:
from rich import print

from stock_search.etf import get_etf_data

print(get_etf_data("VOOG"))

In [ ]:
from rich import print

from stock_search.quote import get_quote

quote = get_quote("AMD")
print(quote)

In [1]:
from rich import print

from stock_search.quote import batch_get_quote

symbols = ["NVDA", "AMD", "GOOG", "BA", "GE", "RTX", "LMT", "VOOG", "AMZN"]
results = batch_get_quote(symbols)

print(results)

Fetching quotes (attempt 2): 100%|██████████| 8/8 [00:06<00:00,  1.29it/s]

✅ Successfully fetched 9/9 quotes


[
    Quote(
        symbol='NVDA',
        regular_price=183.16,
        regular_change=1.01,
        regular_change_percent=0.55,
        realtime_price=182.86,
        realtime_change=-0.3,
        realtime_change_percent=-0.16
    ),
    Quote(
        symbol='AMD',
        regular_price=174.95,
        regular_change=2.67,
        regular_change_percent=1.55,
        realtime_price=175.18,
        realtime_change=0.23,
        realtime_change_percent=0.13
    ),
    Quote(
        symbol='GOOG',
        regular_price=204.16,
        regular_change=2.53,
        regular_change_percent=1.25,
        realtime_price=204.18,
        realtime_change=0.02,
        realtime_change_percent=0.01
    ),
    Quote(
        symbol='BA',
        regular_price=232.61,
        regular_change=6.65,
        regular_change_percent=2.94,
        realtime_price=232.27,
        realtime_change=-0.34,
        realtime_change_percent=-0.15
    ),
    Quote(
        symbol='GE',
        regular_price=279.63,
        regular_change=4.43,
        regular_change_percent=1.61,
        realtime_price=280.01,
        realtime_change=0.38,
        realtime_change_percent=0.14
    ),
    Quote(
        symbol='RTX',
        regular_price=155.49,
        regular_change=0.69,
        regular_change_percent=0.45,
        realtime_price=156.05,
        realtime_change=0.56,
        realtime_change_percent=0.36
    ),
    Quote(
        symbol='LMT',
        regular_price=431.56,
        regular_change=5.3,
        regular_change_percent=1.24,
        realtime_price=430.0,
        realtime_change=-1.56,
        realtime_change_percent=-0.36
    ),
    Quote(
        symbol='VOOG',
        regular_price=417.64,
        regular_change=4.28,
        regular_change_percent=1.04,
        realtime_price=421.0,
        realtime_change=3.36,
        realtime_change_percent=0.8
    ),
    Quote(
        symbol='AMZN',
        regular_price=221.47,
        regular_change=0.17,
        regular_change_percent=0.08,
        realtime_price=221.67,
        realtime_change=0.2,
        realtime_change_percent=0.09
    )
]

In [3]:
from rich import print

from stock_search.indicators import StockIndicator

indicator = StockIndicator("NVDA")

print(indicator.get_all_indicators())

{
    'analyst_rating': 'Strong Buy',
    'earning_direction': 'Increase',
    'eps_growth_percent': 32.903225806451616,
    'expected_earnings_percent': 70.107,
    'fifty_day_change_percent': 13.988492999999998,
    'ma_strategy': '🚀 STRONG BULLISH - All signals aligned, price well above 200MA',
    'market_cap': '4.347333443584B',
    'peg': 2.1536789138576777,
    'two_hundred_day_change_percent': 31.841552,
    'upside_downside': 3.7809940536295272,
    'volume_surge_percent': -5.080645377486526
}

In [ ]:
import yfinance as yf
from rich import print

ticker = yf.Ticker("AMD")
quote = ticker.info

print(quote)

In [ ]:
quote["earningsGrowth"], quote["grossMargins"]

In [ ]:
(quote["forwardEps"] / quote["trailingEps"] - 1) * 100

In [ ]:
quote["beta"]

In [ ]:
# Volume surge %
(quote["volume"] / quote["averageVolume10days"] - 1) * 100

In [ ]:
# Forward P/E can be lower than the trailing P/E if a company is expected to increase its earnings in the coming year, vice versa
# > 0 when analysts expect earnings to grow faster than price
(quote["trailingPE"] - quote["forwardPE"]) / quote["trailingPE"]

In [ ]:
quote["trailingPE"] / (quote["earningsGrowth"] * 100)

In [ ]:
market_cap = quote["marketCap"] / 1e12 if quote["marketCap"] > 1e12 else quote["marketCap"] / 1e9
market_cap

In [ ]:
quote["fiftyDayAverageChangePercent"] * 100

In [ ]:
quote["twoHundredDayAverageChangePercent"] * 100

In [ ]:
"Bullish" if quote["fiftyDayAverageChangePercent"] > 0 and quote["twoHundredDayAverageChangePercent"] > 0 else ("Mixed" if quote["fiftyDayAverageChangePercent"] * quote["twoHundredDayAverageChangePercent"] < 0 else "Bearish")

In [ ]:
quote["fiftyDayAverageChangePercent"] > quote["twoHundredDayAverageChangePercent"]

In [ ]:
# Upside / Downside
(quote["targetMedianPrice"] / quote["currentPrice"] - 1) * 100

In [ ]:
quote["averageAnalystRating"].split(" - ")[1]

In [ ]:
quote["trailingEps"], quote["forwardEps"], quote["earningsGrowth"]

In [ ]:
(quote["forwardEps"] / quote["trailingEps"] - 1)

In [ ]:
quote["grossMargins"]

In [ ]:
(quote["forwardEps"] / quote["trailingEps"] - 1) * quote["grossMargins"]

In [ ]:
from stock_search.utils import datetime_to_str

datetime_to_str(quote["earningsTimestamp"])

In [ ]:
import requests
from rich import print

response = requests.get("https://financialmodelingprep.com/stable/quote?symbol=NVDA&apikey=qHXRYd847oSWxztRTpzGp5XLB5QjbH9B").json()

print(response)

In [ ]:
import yfinance as yf
from rich import print

ticker = yf.Ticker("NVDA")
quote = ticker.info

print(quote)

In [1]:
from rich import print

from stock_search.news import get_news_api, get_news_yfinance

articles = get_news_yfinance("AMD")

len(articles)
print(articles)

[
    News(
        title='AMD (AMD) Q2 Earnings Report Preview: What To Look For',
        url='https://finance.yahoo.com/news/amd-amd-q2-earnings-report-031858481.html',
        content="## AMD (AMD) Q2 Earnings Report Preview: What To Look For\n\nComputer processor maker AMD 
(NASDAQ:AMD) will be reporting earnings this Tuesday after market hours. Here’s what investors should know.\n\nAMD 
beat analysts' revenue expectations by 4.4% last quarter, reporting revenues of $7.44 billion, up 35.9% year on 
year. It was a satisfactory quarter for the company, with an impressive beat of analysts' adjusted operating income
estimates but an increase in its inventory levels.\n\nThis quarter, analysts are expecting AMD’s revenue to grow 
27.3% year on year to $7.43 billion, improving from the 8.9% increase it recorded in the same quarter last year. 
Adjusted earnings are expected to come in at $0.48 per share.\n\nAMD Total Revenue\n\nThe majority of analysts 
covering the company have reconfirmed their estimates over the last 30 days, suggesting they anticipate the 
business to stay the course heading into earnings. AMD has a history of exceeding Wall Street’s expectations, 
beating revenue estimates every single time over the past two years by 1.5% on average.\n\nLooking at AMD’s peers 
in the processors and graphics chips segment, some have already reported their Q2 results, giving us a hint as to 
what we can expect. Qorvo’s revenues decreased 7.7% year on year, beating analysts' expectations by 5.3%, and 
Penguin Solutions reported revenues up 7.9%, falling short of estimates by 1.4%. Qorvo traded up 2.3% following the
results while Penguin Solutions was also up 10.6%.\n\nInvestors in the processors and graphics chips segment have 
had fairly steady hands going into earnings, with share prices down 1.5% on average over the last month. AMD is up 
26.6% during the same time and is heading into earnings with an average analyst price target of $155.96 (compared 
to the current share price of $170.69).",
        sentiment='neutral',
        date='2025-08-04 03:18:58'
    ),
    News(
        title='Dow Jones Futures: Market Tries To Steady After Sell-Off; Palantir, AMD Earnings Ahead',
        url='https://finance.yahoo.com/m/5713215c-df2b-3cad-9b5a-cb6307eeb94f/dow-jones-futures%3A-market.html',
        content="## Dow Jones Futures: Market Tries To Steady After Sell-Off; Palantir, AMD Earnings Ahead\n\nThe 
stock market experienced a sell-off on Friday, influenced by factors such as Trump's tariffs and a disappointing 
jobs report. Investors are now looking ahead to a significant earnings week, with Palantir and AMD set to release 
their results.\n\n**Key Takeaways:**\n\n*   **Market Sell-off:** Friday saw a notable decline in the stock market, 
attributed to the imposition of Trump's tariffs and a weaker-than-expected jobs report.\n*   **Upcoming Earnings:**
This week is packed with important earnings reports, featuring major companies like Palantir and AMD.\n*   
**Investor Strategy:** In light of the recent market volatility, investors are advised to consider their next steps
carefully.",
        sentiment='neutral',
        date='2025-08-04 05:07:58'
    ),
    News(
        title='McDonald’s Earnings, New Tariff Deadline, PMIs: What to Watch This Week',
        url='https://finance.yahoo.com/m/e823428f-68b3-337a-a897-c0f9dce9b878/mcdonald%E2%80%99s-earnings%2C-new.ht
ml',
        content='## McDonald’s Earnings, New Tariff Deadline, PMIs: What to Watch This Week\n\nEarnings season 
marches on, with results from Walt Disney and McDonald’s among this week\'s highlights, along with President 
Trump\'s new tariff-hike deadline for many countries. Here’s what to watch:\n\n**Today**\n\n*   **Economic data:** 
Durable goods and factory orders for June\n*   **Earnings:** Hims & Hers Health, Palantir, Tyson Foods\n*   
**Geopolitics:** Trump administration officials are expected to tout U.S. markets open in 3h 44m US Europe Asia 
Cryptocu

In [ ]:
from rich import print

from stock_search.news import get_news_api, get_news_yfinance

articles = get_news_api("AMD")

len(articles)
print(articles)

[
    News(
        title='Dave Vellante’s Breaking Analysis: The complete collection',
        url='https://siliconangle.com/2025/08/02/dave-vellantes-breaking-analysis-complete-collection/',
        content='### Dave Vellante’s Breaking Analysis: The complete collection\n\n**Publication Date:** Not 
specified\n**Author:** Dave Vellante\n\nBreaking Analysis is a weekly editorial program that combines insights from
theCUBE with spending data from Enterprise Technology Research (ETR). The program, branded as theCUBE Insights, 
Powered by ETR, aims to provide independent, unfiltered editorial content to the SiliconANGLE, theCUBE, and Wikibon
communities. The conclusions drawn are data-driven, leveraging ETR’s proprietary spending data.\n\n**Key Themes and
Episodes:**\n\n*   **AI and Key Players:** Several episodes focus on the expanding breadth of AI, with specific 
attention to Nvidia and Broadcom. Episode 221 highlights Nvidia\'s GTC conference and Broadcom\'s investor day, 
positioning both companies as leaders in the AI era. Other episodes delve into Nvidia\'s market position (Episode 
220), Broadcom\'s strategy (Episode 218), and the impact of AI on cloud providers like AWS (Episode 208, 203, 190, 
189, 188, 181, 177, 154, 152, 141, 140, 137, 134, 131, 128, 117, 110, 109, 108, 104, 103, 98, 94, 92, 89, 86, 84, 
76, 73, 69, 67, 64, 63, 62, 61, 58, 57, 48, 45, 38, 27, 21, 18, 14), Google Cloud (Episode 195, 194, 190, 154, 152,
108, 104, 45, 38), and Microsoft Azure (Episode 206, 203, 190, 154, 152, 108, 57, 38).\n*   **Cybersecurity:** The 
cybersecurity landscape is frequently discussed, with a focus on companies like CrowdStrike (Episode 219, 197, 147,
130, 119, 49, 42, 16) and Palo Alto Networks (Episode 219, 159, 130, 97, 70, 49, 42, 16). The impact of AI on 
cybersecurity is also a recurring topic (Episode 189).\n*   **Data Platforms and Architecture:** The evolution of 
data platforms, including the concept of the "Sixth Data Platform," is explored in episodes like 215, 209, 201, 
187, 186, 183, 181, 170, 143, 141, 117, 116, 113, 105, 91, 5, 3.\n*   **Enterprise Technology Spending:** Numerous 
episodes analyze IT spending trends, often referencing ETR data, to provide insights into market conditions, 
company performance, and future outlooks (e.g., Episode 214, 213, 200, 193, 192, 191, 189, 180, 179, 178, 177, 176,
175, 174, 173, 172, 171, 168, 167, 166, 165, 164, 163, 162, 161, 160, 159, 158, 157, 156, 155, 154, 153, 152, 151, 
150, 149, 148, 147, 146, 145, 144, 143, 142, 141, 140, 139, 138, 137, 136, 135, 134, 133, 132, 131, 130, 129, 128, 
127, 126, 125, 124, 123, 122, 121, 120, 119, 118, 117, 116, 115, 114, 113, 112, 111, 110, 109, 108, 107, 106, 105, 
104, 103, 102, 101, 100, 99, 98, 97, 96, 95, 94, 93, 92, 91, 90, 89, 88, 87, 86, 85, 84, 83, 82, 81, 80, 79, 78, 
77, 76, 75, 74, 73, 72, 71, 70, 69, 68, 67, 66, 65, 64, 63, 62, 61, 60, 59, 58, 56, 55, 54, 53, 52, 51, 50, 49, 48,
47, 46, 45, 44, 43, 42, 41, 40, 39, 38, 36, 35, 34, 33, 32, 31, 30, 29, 28, 27, 26, 25, 24, 23, 22, 21, 20, 19, 18,
17, 16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1). Specific topics include the impact of macroeconomics, 
cloud optimization, generative AI, cybersecurity, and the performance of various tech companies.\n*   
**Company-Specific Analysis:** Deep dives into companies like Nvidia, Broadcom, CrowdStrike, Palo Alto Networks, 
IBM, Intel, Cisco, UiPath, Snowflake, Databricks, HPE, and others are provided, often with a focus on their 
strategies, market positioning, and financial performance.\n\nThe collection represents a comprehensive overview of
the enterprise technology landscape, with a strong emphasis on the transformative impact of AI and cloud computing,
supported by rigorous data analysis from ETR.',
        sentiment='neutral',
        date='2025-08-02 10:33:12'
    ),
    News(
        title='Analysts Set Advanced Micro Devices, Inc. (NASDAQ:AMD) Price Target at $162.94',
        url='https://www.etfdailynews.com/2